# Matrix Multiplication

Addition and scaling treat a matrix as a bag of numbers sitting in a grid. Multiplication
is where a matrix starts *doing* something: it takes a vector in and hands a different
vector back. Every quantum gate you will ever meet is a small matrix, and applying a gate
to a qubit is exactly the product you are about to work out on paper.

This is the heaviest practice set in the module, and that is deliberate. The rule itself
is one sentence long -- one dot product per row-and-column pair -- but the bookkeeping is
where people lose whole afternoons later. Twelve exercises here buy you a lifetime of
reading circuit algebra without flinching.

Do every product by hand first. NumPy arrives at the end, once the arithmetic is yours,
and it arrives carrying the one trap that catches everybody.

**Objectives:**
- Multiply a matrix by a column vector, one dot product per row
- Multiply two matrices column by column, with every term written out longhand
- Apply the shape rule, and say when a product is simply undefined
- Show that order matters, that grouping does not, and what the identity and zero matrices do
- Use `@` for the matrix product and know why `*` is a different operation
- Tell a 1-D NumPy vector from a 2-D column, and predict what `@` returns for each

**Reference:** See [`../GUIDE.md`](../GUIDE.md).

<!-- browser-runnable -->


In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)


## 1. A matrix times a column vector

Start with the smaller of the two products, because the bigger one is built entirely out
of it.

A matrix has rows. A column vector has entries. To multiply them, take the first row of
the matrix, pair it up with the vector entry by entry, multiply each pair, and add the
results. That single number is the first entry of the answer. Do the same with the second
row to get the second entry, and so on down the matrix.

One row in, one number out. A matrix with two rows therefore returns a vector with two
entries, whatever else the matrix looks like.

Two special cases are worth meeting immediately, because both come back later wearing a
quantum gate's name tag.

- The **identity matrix** has ones down the diagonal and zeros everywhere else. Its first
  row is `[1, 0]`, so the first entry of the answer is the vector's first entry and
  nothing else. Every row behaves the same way. The identity hands the vector back exactly
  as it found it: it is the do-nothing matrix.
- The **swap matrix** `[[0, 1], [1, 0]]` carries its ones off the diagonal. Its first row
  `[0, 1]` ignores the vector's first entry and picks out the second; its second row does
  the reverse. The swap matrix exchanges the two entries.

Watch both of them work below, and then a matrix with no zeros in it at all.


In [ ]:
v = np.array([5, -1])

I2 = np.array([[1, 0],
               [0, 1]])
S = np.array([[0, 1],
              [1, 0]])
A = np.array([[3, 1],
              [2, 4]])

print("v      =", v)
print("I2 @ v =", I2 @ v)     # unchanged: the do-nothing matrix
print("S  @ v =", S @ v)      # the two entries have traded places
print("A  @ v =", A @ v)      # row 1: 3*5 + 1*(-1),  row 2: 2*5 + 4*(-1)


In symbols, if $A$ is a matrix and $x$ is a column vector, the $i$-th entry of the product
is

$$(Ax)_i = \sum_j A_{ij} x_j$$

Read that sum as the sentence you just carried out: run along row $i$ of $A$ and down the
vector $x$ together, multiply the pairs, add them up. The index $j$ is the one doing the
running, and it is gone by the time the entry is written down.

In NumPy the operator is `@`. It is not `*`. Section 5 is about exactly that difference,
and it is the single most common bug in this material.


## 2. A matrix times a matrix

Now the general product. Here is the sentence that makes it easy to remember:

**Each column of the answer is the left matrix times the corresponding column of the right
matrix.**

That is the whole definition. You already know how to multiply a matrix by a column
vector, so a matrix product is just that job done once per column of the right-hand
matrix, with the answers set side by side.

Take

$$A = \begin{pmatrix} 2 & 1 \\ 0 & 3 \end{pmatrix}, \qquad
B = \begin{pmatrix} 1 & 4 \\ 5 & 2 \end{pmatrix}$$

`B` has two columns, so the product has two columns. Nothing below is skipped.

Column 1 of the answer is `A` times B's first column, `[1, 5]`:

- top entry: `2*1 + 1*5 = 7`
- bottom entry: `0*1 + 3*5 = 15`

Column 2 of the answer is `A` times B's second column, `[4, 2]`:

- top entry: `2*4 + 1*2 = 10`
- bottom entry: `0*4 + 3*2 = 6`

Stand the two columns side by side and the product is

$$AB = \begin{pmatrix} 7 & 10 \\ 15 & 6 \end{pmatrix}$$

Four entries, four dot products, all four written out longhand. Once you trust the
pattern, the usual shortcut is to read the answer entry by entry rather than column by
column: the entry in row $i$, column $j$ is row $i$ of the left matrix dotted with column
$j$ of the right one. Same arithmetic, different order of bookkeeping.


In [ ]:
A2 = np.array([[2, 1],
               [0, 3]])
B2 = np.array([[1, 4],
               [5, 2]])

col1 = A2 @ B2[:, 0]          # A2 times B2's first column
col2 = A2 @ B2[:, 1]          # A2 times B2's second column
print("column 1 of the answer:", col1)
print("column 2 of the answer:", col2)

print("\nA2 @ B2 =")
print(A2 @ B2)                # the same two columns, stood side by side


In symbols,

$$(AB)_{ij} = \sum_k A_{ik} B_{kj}$$

The repeated index $k$ runs along a row of $A$ and down a column of $B$ at the same time,
which is exactly the pairing you did by hand. The indices that survive, $i$ and $j$, are
the ones naming the answer's position. The fact that $k$ vanishes into the sum is the
whole reason the next section's shape rule looks the way it does.


## 3. Shapes, and when there is no product at all

Every dot product in that definition pairs a row of the left matrix against a column of
the right one, term by term. That only works if the row and the column are the same
length. So:

**A row of the left matrix has one entry per column of the left matrix. A column of the
right matrix has one entry per row of the right matrix. Those two counts must match.**

Written out, an $m \times n$ matrix times an $n \times p$ matrix is defined, and gives an
$m \times p$ matrix:

$$(m \times n)(n \times p) = (m \times p)$$

Two things to take from that line. The **inner** dimensions must agree, and then they
vanish -- the shared $n$ is what got summed over. The **outer** dimensions survive
untouched: rows of the left, columns of the right.

If the inner dimensions disagree there is no product. Not a zero, not an error to patch
around: the operation is undefined, and NumPy raises a `ValueError` rather than guessing
what you meant.

Rectangular matrices make the point loudly. A `(2, 3)` matrix times a `(3, 2)` matrix is
`(2, 2)`. Turn the same two matrices around and a `(3, 2)` times a `(2, 3)` is `(3, 3)`.
Same two matrices, two perfectly legal products, and not even the same size of answer.


In [ ]:
R23 = np.array([[1, 0, 2],
                [3, 1, 1]])        # 2 rows, 3 columns
C32 = np.array([[1, 2],
                [0, 1],
                [4, 0]])           # 3 rows, 2 columns

print("R23.shape:", R23.shape, " C32.shape:", C32.shape)

print("\nR23 @ C32 has shape", (R23 @ C32).shape)
print(R23 @ C32)

print("\nC32 @ R23 has shape", (C32 @ R23).shape)
print(C32 @ R23)

try:
    R23 @ R23                      # inner dimensions 3 and 2 disagree
except ValueError as err:
    print("\nR23 @ R23 is undefined ->", err)


## 4. What the product obeys, and what it does not

Ordinary numbers let you shuffle a product around freely. Matrices do not, and the one
rule they break is the most consequential fact in this notebook.

**Order matters.** In general $AB \ne BA$. Sometimes only the entries differ. With
rectangular matrices even the shapes differ, as you just saw. Sometimes one order is
defined and the other is not defined at all. Two matrices that happen to satisfy
$AB = BA$ are said to **commute**, and commuting is a special property, not the norm.

**Grouping does not matter.** $(AB)C = A(BC)$, always, whenever the shapes line up. This
is **associativity**, and it is what lets you write $ABC$ with no brackets. The order is
fixed; where you put the parentheses is free.

**The identity is the multiplicative one.** $IA = AI = A$ for every $A$ of the matching
size, exactly as multiplying a number by one changes nothing.

**The zero matrix is the multiplicative zero.** All zeros times anything is all zeros.
Note one asymmetry with ordinary numbers, though: two nonzero matrices can multiply to
zero, which never happens with two nonzero numbers.


In [ ]:
print("A @ S =")
print(A @ S)                  # S on the right rearranges A's columns
print("S @ A =")
print(S @ A)                  # S on the left rearranges A's rows
print("do they commute?", np.array_equal(A @ S, S @ A))

print("\nassociativity -- (A S) B2 equals A (S B2)?",
      np.array_equal((A @ S) @ B2, A @ (S @ B2)))

Z2 = np.zeros((2, 2), dtype=int)
print("\nI2 @ A leaves A alone:", np.array_equal(I2 @ A, A))
print("Z2 @ A is all zeros:  ", np.array_equal(Z2 @ A, Z2))

N1 = np.array([[0, 1],        # nonzero...
               [0, 0]])
print("\nN1 @ N1 =")
print(N1 @ N1)                # ...yet its square is the zero matrix


## 5. In NumPy, `@` is the product and `*` is not

NumPy gives you two multiplication operators and they mean different things.

- `A @ B` is the matrix product of this notebook: rows against columns, sums of products,
  shapes governed by the rule in Section 3.
- `A * B` multiplies **entry by entry**. The value in row $i$, column $j$ of the result is
  just `A[i, j] * B[i, j]`. No sums, no dot products, and both matrices must already have
  the same shape.

Both are legal Python, both return an array, and neither one warns you. On two matrices of
the same shape they quietly produce different answers, which is why a stray `*` can
survive a long way into a calculation before anything looks broken. When you mean the
matrix product, type `@`.

There is a second trap underneath the first: **NumPy has two different things that both
look like a vector.**

- `np.array([5, -1])` is **1-D**, with shape `(2,)`. It is neither a row nor a column
  until an operator decides for it.
- `np.array([[5], [-1]])` is **2-D**, with shape `(2, 1)`. That one is genuinely a column.

`@` reads a 1-D array as whichever orientation makes the product defined: as a column when
it is on the right, as a row when it is on the left. So for a 1-D `x`, both `A @ x` and
`x @ A` run without complaint and compute different things. Hand `@` a real 2-D column
instead and the shape rule applies as written, and the answer comes back 2-D as well.

Printing `.shape` costs one line and settles the question every time.


In [ ]:
print("A @ B2 (matrix product) =")
print(A @ B2)
print("A * B2 (entry by entry) =")
print(A * B2)

x1 = np.array([5, -1])            # 1-D, shape (2,)
x2 = np.array([[5],
               [-1]])             # 2-D column, shape (2, 1)
print("\nx1.shape:", x1.shape, "  x2.shape:", x2.shape)

print("A @ x1 =", A @ x1, " shape", (A @ x1).shape)   # x1 read as a column
print("x1 @ A =", x1 @ A, " shape", (x1 @ A).shape)   # x1 read as a row: different answer
print("A @ x2 =")
print(A @ x2)                                          # still 2-D, shape (2, 1)


## 6. Notation cheat sheet

| Math | NumPy | Meaning |
|---|---|---|
| $Ax$ | `A @ x` | Matrix times column vector: one dot product per row |
| $AB$ | `A @ B` | Matrix product: each column is `A` times a column of `B` |
| $(AB)_{ij} = \sum_k A_{ik} B_{kj}$ | `(A @ B)[i, j]` | Row $i$ of `A` against column $j$ of `B` |
| $(m \times n)(n \times p)$ | `A.shape`, `B.shape` | Defined only when the inner dimensions match |
| $I$ | `np.eye(n)` | The do-nothing matrix, with $IA = AI = A$ |
| entry by entry | `A * B` | Not the matrix product: no sums are involved |

Four habits, in rough order of how much time they save:

1. **Check the shapes before you multiply.** Most "wrong answer" bugs in this material are
   really "undefined product that I forced through by transposing something".
2. **Never type `*` when you mean `@`.** Same shapes in, same shape out, silently
   different numbers.
3. **Keep the order.** $AB$ and $BA$ are different questions, and swapping them is not a
   typo the mathematics will quietly forgive.
4. **Print `.shape`** whenever a result surprises you, especially with 1-D vectors, where
   `@` picks the orientation on your behalf.


## 7. Exercises

Twelve exercises follow. The first nine are pure paper arithmetic, because that is the only
way this becomes automatic; the last three turn to shapes, order and NumPy.

Before them, one tool for unlimited extra practice. `drill` generates fresh problems of the
same kinds this notebook taught and marks your answer without ever showing it to you:
`check()` reports which entries are off and by how much, and `reveal()` is the only door to
the answer. Open it after you have tried, not before.

- `"matvec"` -- a matrix times a column vector (Section 1)
- `"multiply"` -- a matrix times a matrix (Section 2)
- `"defined"` -- given two shapes, is the product defined, and what shape does it have (Section 3)

Levels run 1 to 3: level 1 is small non-negative 2x2 work, level 2 brings in negatives and
3x3, and level 3 goes rectangular and includes pairs whose product is deliberately
undefined. Leave `seed=` off and the module draws one for you and prints it, so any problem
worth returning to is one number away.


In [ ]:
from lib.linalg_drills import drill

d = drill("matvec", level=1, seed=18)
d.show()

# Work it on paper, then hand your answer over -- a plain list is fine:
#     d.check([...])
# and once you have genuinely tried:
#     d.reveal()


### Exercise 1 — The identity leaves a vector alone

Multiply the 2x2 identity matrix by the vector below **by hand**, one dot product per row,
and write the result down.

$$I = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}, \qquad
x = \begin{pmatrix} 4 \\ -3 \end{pmatrix}$$

Define `ex1_iv` -- the product $Ix$, as a flat two-entry sequence (a Python list or a 1-D
NumPy array, not a column of one-entry rows).

<details><summary>Hint 1 — nudge</summary>

Each entry of the answer is one row of the matrix paired against the whole vector. Look at
what the identity's first row is made of: against a row that is mostly zeros, how many of
the vector's entries can survive the sum?

</details>
<details><summary>Hint 2 — approach</summary>

Write each dot product out in full before you evaluate it: row 1's two entries against the
vector's two entries, multiplied pairwise and added. Repeat for row 2, then state the
answer as a two-entry list.

</details>


In [ ]:
# Exercise 1: Multiply the identity matrix by the vector (4, -3) by hand.
# Define: ex1_iv -- the product, as a flat two-entry sequence.

# TODO: your code here


In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    assert np.asarray(ex1_iv).shape == (2,), (
        "a 2x2 matrix times a 2-entry vector gives a 2-entry vector -- write it flat, "
        "not as a column of one-entry rows"
    )
    assert np.allclose(np.asarray(ex1_iv), np.array([[1, 0], [0, 1]]) @ np.array([4, -3])), (
        "re-do each entry as a dot product: a row with a single 1 in it can only pick up "
        "one of the vector's entries"
    )


### Exercise 2 — The swap matrix exchanges the entries

Same vector, different matrix. Multiply **by hand**:

$$S = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \qquad
x = \begin{pmatrix} 4 \\ -3 \end{pmatrix}$$

Define `ex2_sv` -- the product $Sx$, again as a flat two-entry sequence.

<details><summary>Hint 1 — nudge</summary>

Section 1 called this the swap matrix for a reason, but do not take the name on trust.
Each row still has exactly one 1 in it -- what changed from Exercise 1 is *where* that 1
sits, and therefore which entry of the vector it lands on.

</details>
<details><summary>Hint 2 — approach</summary>

Do the two dot products in the same order as before: row 1 against the vector, then row 2
against the vector. Multiply the zero terms out explicitly rather than skipping them; that
habit is what keeps 3x3 work honest later.

</details>


In [ ]:
# Exercise 2: Multiply the swap matrix [[0, 1], [1, 0]] by (4, -3) by hand.
# Define: ex2_sv -- the product, as a flat two-entry sequence.

# TODO: your code here


In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    assert np.asarray(ex2_sv).shape == (2,), (
        "the answer has one entry per row of the matrix, written as a flat sequence"
    )
    assert np.allclose(np.asarray(ex2_sv), np.array([[0, 1], [1, 0]]) @ np.array([4, -3])), (
        "check the signs: one of the vector's entries is negative, and it does not stay "
        "where it started"
    )


### Exercise 3 — A matrix with nothing to hide behind

No convenient zeros this time. Multiply **by hand**:

$$N = \begin{pmatrix} 2 & -1 \\ 3 & 4 \end{pmatrix}, \qquad
u = \begin{pmatrix} 5 \\ 2 \end{pmatrix}$$

Define `ex3_mv` -- the product $Nu$, as a flat two-entry sequence.

<details><summary>Hint 1 — nudge</summary>

Nothing new is being asked. The same rule from Exercises 1 and 2 applies; the only reason
this one feels harder is that every term in both sums is now doing real work, so nothing
can be skipped.

</details>
<details><summary>Hint 2 — approach</summary>

For the first entry, pair row 1 with the vector: first-with-first, second-with-second,
multiply, add. Watch the negative entry -- it subtracts. Then do row 2 the same way, and
sanity-check that you used the vector twice and each matrix row exactly once.

</details>


In [ ]:
# Exercise 3: Multiply [[2, -1], [3, 4]] by the vector (5, 2) by hand.
# Define: ex3_mv -- the product, as a flat two-entry sequence.

# TODO: your code here


In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    assert np.asarray(ex3_mv).shape == (2,), (
        "two rows in the matrix means two entries in the answer"
    )
    assert np.allclose(np.asarray(ex3_mv), np.array([[2, -1], [3, 4]]) @ np.array([5, 2])), (
        "re-check the first entry: row 1 of the matrix against the vector, term by term, "
        "and mind the minus sign"
    )


### Exercise 4 — Two by two, column by column

Your first full matrix product, **by hand**:

$$J = \begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix}, \qquad
K = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$$

Define `ex4_product` -- the product $JK$, as a 2x2 nested list or array.

<details><summary>Hint 1 — nudge</summary>

Section 2's sentence is the whole method: each column of the answer is the left matrix
times the corresponding column of the right matrix. So this is two matrix-times-vector
problems of the kind you have already done three times, stood side by side.

</details>
<details><summary>Hint 2 — approach</summary>

Read K's first column off vertically, multiply J by it as in Exercise 3, and that is
column 1 of the answer. Repeat with K's second column for column 2. When you assemble the
nested list, remember you are writing it out by rows even though you computed it by
columns.

</details>


In [ ]:
# Exercise 4: Compute J @ K by hand for J = [[1, 2], [3, 4]], K = [[0, 1], [1, 0]].
# Define: ex4_product -- the 2x2 product, as a nested list or array.

# TODO: your code here


In [ ]:
# Check Exercise 4 -- run after your attempt.
from lib.grading import check

with check("Exercise 4"):
    assert np.asarray(ex4_product).shape == (2, 2), (
        "a 2x2 times a 2x2 is 2x2 -- the answer needs two rows of two entries"
    )
    assert np.allclose(
        np.asarray(ex4_product),
        np.array([[1, 2], [3, 4]]) @ np.array([[0, 1], [1, 0]]),
    ), (
        "re-check the top-left entry: row 1 of J against column 1 of K. If your answer "
        "looks like J with its rows rearranged, you multiplied in the other order"
    )


### Exercise 5 — Two by two with a zero in the way

**By hand** again:

$$P = \begin{pmatrix} 2 & 0 \\ 1 & 3 \end{pmatrix}, \qquad
Q = \begin{pmatrix} 1 & 4 \\ 2 & -1 \end{pmatrix}$$

Define `ex5_product` -- the product $PQ$, as a 2x2 nested list or array.

You will meet $P$ and $Q$ once more in Exercise 12, so keep your answer where you can find
it.

<details><summary>Hint 1 — nudge</summary>

A zero in the left matrix kills one term of every sum that uses it -- and its row is used
once per column of the right matrix, so that is two sums here, not one. It still does not
excuse you from writing them down: every entry of the answer is a two-term dot product.

</details>
<details><summary>Hint 2 — approach</summary>

Four entries, four dot products. Take them in reading order: row 1 with column 1, row 1
with column 2, row 2 with column 1, row 2 with column 2. Write each as `a*c + b*d` before
you evaluate it, and keep track of the one negative entry in Q.

</details>


In [ ]:
# Exercise 5: Compute P @ Q by hand for P = [[2, 0], [1, 3]], Q = [[1, 4], [2, -1]].
# Define: ex5_product -- the 2x2 product, as a nested list or array.

# TODO: your code here


In [ ]:
# Check Exercise 5 -- run after your attempt.
from lib.grading import check

with check("Exercise 5"):
    assert np.asarray(ex5_product).shape == (2, 2), (
        "the product of two 2x2 matrices is 2x2"
    )
    assert np.allclose(
        np.asarray(ex5_product),
        np.array([[2, 0], [1, 3]]) @ np.array([[1, 4], [2, -1]]),
    ), (
        "P's zero sits in its TOP row, so both top-row sums lose their second term and "
        "the answer's top row is just twice the top row of Q -- re-check that row first, "
        "then walk row 2 of P against each column of Q in turn"
    )


### Exercise 6 — Two by two with negatives throughout

The last of the 2x2 hand products, and the least forgiving:

$$E = \begin{pmatrix} 3 & -1 \\ -2 & 4 \end{pmatrix}, \qquad
F = \begin{pmatrix} 2 & 5 \\ 1 & 0 \end{pmatrix}$$

Define `ex6_product` -- the product $EF$, as a 2x2 nested list or array.

<details><summary>Hint 1 — nudge</summary>

Nothing about the rule changes when entries go negative. What changes is that a sign slip
now produces a plausible-looking wrong answer instead of an obviously wrong one, so the
arithmetic deserves a second pass.

</details>
<details><summary>Hint 2 — approach</summary>

Write every term with its sign attached before adding: a negative times a positive is
negative, and a negative times a negative is positive. One entry of this answer comes out
to zero and another comes out negative -- if neither happens for you, the signs are worth
re-reading.

</details>


In [ ]:
# Exercise 6: Compute E @ F by hand for E = [[3, -1], [-2, 4]], F = [[2, 5], [1, 0]].
# Define: ex6_product -- the 2x2 product, as a nested list or array.

# TODO: your code here


In [ ]:
# Check Exercise 6 -- run after your attempt.
from lib.grading import check

with check("Exercise 6"):
    assert np.asarray(ex6_product).shape == (2, 2), (
        "two rows, two columns -- the shape rule does not care about signs"
    )
    assert np.allclose(
        np.asarray(ex6_product),
        np.array([[3, -1], [-2, 4]]) @ np.array([[2, 5], [1, 0]]),
    ), (
        "walk the four dot products again with the signs written in: the entries of E "
        "carry them, and F's single zero sits at the bottom of its second column, so BOTH "
        "entries in the answer's second column collapse to one surviving term"
    )


### Exercise 7 — Three by three, times a column

Same rule, one size up. **By hand**:

$$T = \begin{pmatrix} 1 & 0 & 2 \\ 0 & 3 & -1 \\ 4 & 1 & 0 \end{pmatrix}, \qquad
w = \begin{pmatrix} 2 \\ -1 \\ 3 \end{pmatrix}$$

Define `ex7_tw` -- the product $Tw$, as a flat three-entry sequence.

<details><summary>Hint 1 — nudge</summary>

The row count of the matrix decides the length of the answer, and the column count decides
how many terms are in each sum. Say both numbers out loud before you start, so you know
what you are aiming at.

</details>
<details><summary>Hint 2 — approach</summary>

Three rows, so three dot products, each with three terms. Keep the vector fixed in front
of you and slide down the matrix one row at a time. Writing the zero terms out explicitly
costs nothing and is what stops a row from quietly losing a term.

</details>


In [ ]:
# Exercise 7: Compute T @ w by hand for the 3x3 matrix T and the 3-entry vector w.
# Define: ex7_tw -- the product, as a flat three-entry sequence.

# TODO: your code here


In [ ]:
# Check Exercise 7 -- run after your attempt.
from lib.grading import check

with check("Exercise 7"):
    assert np.asarray(ex7_tw).shape == (3,), (
        "three rows in the matrix means three entries in the answer, written flat"
    )
    assert np.allclose(
        np.asarray(ex7_tw),
        np.array([[1, 0, 2], [0, 3, -1], [4, 1, 0]]) @ np.array([2, -1, 3]),
    ), (
        "each entry is a three-term sum -- re-check the middle row, where both nonzero "
        "terms pull in the same direction"
    )


### Exercise 8 — Three by three, times three by three

Nine entries, nine dot products, three terms each. **By hand**:

$$G = \begin{pmatrix} 1 & 0 & 1 \\ 0 & 2 & 0 \\ 1 & 1 & 0 \end{pmatrix}, \qquad
H = \begin{pmatrix} 2 & 1 & 0 \\ 0 & 1 & 3 \\ 1 & 0 & 1 \end{pmatrix}$$

Define `ex8_product` -- the product $GH$, as a 3x3 nested list or array.

<details><summary>Hint 1 — nudge</summary>

This is Exercise 7 done three times: each column of the answer is $G$ times the
corresponding column of $H$. The entries are small on purpose, so the only real difficulty
is not losing your place.

</details>
<details><summary>Hint 2 — approach</summary>

Pick one bookkeeping order and hold it for all nine entries -- column by column is the
safest, because each column is one matrix-times-vector problem you already know how to do.
Then transcribe the three columns into rows when you write the nested list. G's middle row
has a single nonzero entry, which makes the middle row of the answer a cheap way to check
yourself.

</details>


In [ ]:
# Exercise 8: Compute G @ H by hand for the two 3x3 matrices above.
# Define: ex8_product -- the 3x3 product, as a nested list or array.

# TODO: your code here


In [ ]:
# Check Exercise 8 -- run after your attempt.
from lib.grading import check

with check("Exercise 8"):
    assert np.asarray(ex8_product).shape == (3, 3), (
        "a 3x3 times a 3x3 is 3x3 -- nine entries, arranged in three rows"
    )
    assert np.allclose(
        np.asarray(ex8_product),
        np.array([[1, 0, 1], [0, 2, 0], [1, 1, 0]]) @ np.array([[2, 1, 0], [0, 1, 3], [1, 0, 1]]),
    ), (
        "compare your middle row against G's middle row: with only one nonzero entry "
        "there, that row of the answer should be a simple multiple of a row of H"
    )


### Exercise 9 — Both orders, two different shapes

Here is a rectangular pair. Both orders are defined, and they do not even produce answers
of the same size. Compute **both, by hand**:

$$U = \begin{pmatrix} 1 & 2 & 0 \\ 0 & 1 & 3 \end{pmatrix}, \qquad
V = \begin{pmatrix} 1 & 0 \\ 2 & 1 \\ 0 & 4 \end{pmatrix}$$

Define `ex9_uv` -- the product $UV$ -- and `ex9_vu` -- the product $VU$. Work out each
one's shape from the rule in Section 3 before you compute a single entry.

<details><summary>Hint 1 — nudge</summary>

$U$ is `(2, 3)` and $V$ is `(3, 2)`. Feed each order through the shape rule: the inner
dimensions have to match, they disappear, and the outer ones survive. The two orders give
you two different sizes of answer, which is already the point of the exercise.

</details>
<details><summary>Hint 2 — approach</summary>

For $UV$, each entry is a three-term sum, because the shared inner dimension is 3. For
$VU$, each entry is a two-term sum. Do the smaller product first to warm up, then take the
larger one column by column and count your terms as you go -- a sum with the wrong number
of terms in it is the classic rectangular mistake.

</details>


In [ ]:
# Exercise 9: Compute both U @ V and V @ U by hand for the rectangular pair above.
# Define: ex9_uv and ex9_vu -- the two products, as nested lists or arrays.

# TODO: your code here


In [ ]:
# Check Exercise 9 -- run after your attempt.
from lib.grading import check

with check("Exercise 9"):
    assert np.asarray(ex9_uv).shape == (2, 2), (
        "for U V the outer dimensions survive: rows of U, columns of V"
    )
    assert np.asarray(ex9_vu).shape == (3, 3), (
        "for V U the outer dimensions are different ones -- rows of V, columns of U"
    )
    assert np.allclose(
        np.asarray(ex9_uv),
        np.array([[1, 2, 0], [0, 1, 3]]) @ np.array([[1, 0], [2, 1], [0, 4]]),
    ), (
        "each entry of U V is a three-term sum -- re-check the bottom-right one: it pairs "
        "U's second row against V's second column, and its FIRST term drops out, so the "
        "other two carry the whole entry"
    )
    assert np.allclose(
        np.asarray(ex9_vu),
        np.array([[1, 0], [2, 1], [0, 4]]) @ np.array([[1, 2, 0], [0, 1, 3]]),
    ), (
        "each entry of V U is only a two-term sum -- if you used three terms anywhere, "
        "you multiplied in the other order"
    )


### Exercise 10 — Which products are defined

No arithmetic at all. For each pair of shapes below, decide whether the product (left
matrix first) exists, and if it does, what shape the answer has.

| Label | Left shape | Right shape |
|---|---|---|
| `"a"` | `(2, 3)` | `(3, 4)` |
| `"b"` | `(3, 4)` | `(2, 3)` |
| `"c"` | `(4, 1)` | `(1, 4)` |
| `"d"` | `(1, 4)` | `(4, 1)` |
| `"e"` | `(2, 2)` | `(3, 3)` |

Define `ex10_shapes` -- a dictionary with those five string keys, where each value is the
result's shape as a two-number tuple, or `None` if the product is undefined.

<details><summary>Hint 1 — nudge</summary>

One question decides every row: does the left matrix's column count equal the right
matrix's row count? Everything else follows mechanically from the rule in Section 3.

</details>
<details><summary>Hint 2 — approach</summary>

Write each pair as two shapes side by side and look only at the two middle numbers. If
they disagree, the entry is `None` and you are done with that row. If they agree, the
answer's shape is built from the two numbers you did not compare. Rows `"c"` and `"d"` use
the same two matrices in opposite orders, so expect their answers to look nothing alike.

</details>


In [ ]:
# Exercise 10: Decide which of the five shape pairs give a defined product, and what shape.
# Define: ex10_shapes -- a dict keyed "a" to "e", each value a (rows, cols) tuple or None.

# TODO: your code here


In [ ]:
# Check Exercise 10 -- run after your attempt.
from lib.grading import check

with check("Exercise 10"):
    # The five shape pairs exactly as the prompt's table lists them. The answers
    # are not written down here: NumPy applies the rule to a blank matrix of each
    # shape, so this cell states the question and never the answer sheet.
    pairs10 = {
        "a": ((2, 3), (3, 4)),
        "b": ((3, 4), (2, 3)),
        "c": ((4, 1), (1, 4)),
        "d": ((1, 4), (4, 1)),
        "e": ((2, 2), (3, 3)),
    }
    expected10 = {}
    for key10, (left10, right10) in pairs10.items():
        try:
            expected10[key10] = (np.zeros(left10) @ np.zeros(right10)).shape
        except ValueError:
            expected10[key10] = None

    assert isinstance(ex10_shapes, dict), (
        "ex10_shapes should be a dictionary, one entry per row of the table"
    )
    assert set(ex10_shapes) == set(pairs10), (
        "use exactly the five string keys from the table's Label column"
    )
    seen10 = {k: (None if val is None else tuple(val)) for k, val in ex10_shapes.items()}
    assert sorted(k for k, val in seen10.items() if val is None) == sorted(
        k for k, val in expected10.items() if val is None
    ), (
        "the rows you marked None are not the rows whose inner dimensions actually "
        "disagree -- compare the left factor's column count with the right factor's row "
        "count once more, row by row"
    )
    for key10 in sorted(pairs10):
        if expected10[key10] is None:
            continue
        assert seen10[key10] == expected10[key10], (
            f"row {key10!r} has a defined product, but its shape is off: the inner "
            "dimension is summed away and the outer two survive -- rows come from the "
            "left factor, columns from the right"
        )


### Exercise 11 — Order matters

Both orders are defined here and both answers are 2x2, so nothing about the shapes warns
you. Compute **both, by hand**:

$$C = \begin{pmatrix} 1 & 1 \\ 0 & 1 \end{pmatrix}, \qquad
D = \begin{pmatrix} 1 & 0 \\ 1 & 1 \end{pmatrix}$$

Define `ex11_cd` -- the product $CD$ -- and `ex11_dc` -- the product $DC$. Then state your
conclusion: define `ex11_commute` as `True` if the two products came out identical and
`False` if they did not.

<details><summary>Hint 1 — nudge</summary>

In $CD$ the left factor supplies the rows and the right factor supplies the columns. Swap
the factors and both jobs change hands, so there is no reason to expect the same answer --
and Section 4 says you should expect a different one.

</details>
<details><summary>Hint 2 — approach</summary>

Compute $CD$ completely, then start $DC$ from scratch rather than trying to adapt the
first answer. Compare the two entry by entry, and let `ex11_commute` record what you
actually found rather than what you assumed.

</details>


In [ ]:
# Exercise 11: Compute C @ D and D @ C by hand, then say whether they are equal.
# Define: ex11_cd, ex11_dc -- the two 2x2 products -- and ex11_commute (True or False).

# TODO: your code here


In [ ]:
# Check Exercise 11 -- run after your attempt.
from lib.grading import check

with check("Exercise 11"):
    assert np.asarray(ex11_cd).shape == (2, 2) and np.asarray(ex11_dc).shape == (2, 2), (
        "both orders are defined here and both answers are 2x2"
    )
    assert np.allclose(
        np.asarray(ex11_cd),
        np.array([[1, 1], [0, 1]]) @ np.array([[1, 0], [1, 1]]),
    ), (
        "for C D, C supplies the rows and D supplies the columns -- re-check the top-left "
        "entry with that assignment"
    )
    assert np.allclose(
        np.asarray(ex11_dc),
        np.array([[1, 0], [1, 1]]) @ np.array([[1, 1], [0, 1]]),
    ), (
        "for D C the two roles trade places: D supplies the rows now"
    )
    assert isinstance(ex11_commute, (bool, np.bool_)), (
        "ex11_commute should be a plain True or False"
    )
    assert bool(ex11_commute) == bool(
        np.allclose(np.asarray(ex11_cd), np.asarray(ex11_dc))
    ), (
        "your verdict should agree with your own two products -- compare them entry by entry"
    )
    assert not ex11_commute, (
        "if your two products came out identical, one of the eight dot products has gone "
        "astray -- redo D C from scratch rather than editing C D"
    )


### Exercise 12 — Let NumPy referee

Time to hand the arithmetic over. Build the three matrices

$$P = \begin{pmatrix} 2 & 0 \\ 1 & 3 \end{pmatrix}, \qquad
Q = \begin{pmatrix} 1 & 4 \\ 2 & -1 \end{pmatrix}, \qquad
R = \begin{pmatrix} 1 & 1 \\ 0 & 2 \end{pmatrix}$$

as `ex12_p`, `ex12_q` and `ex12_r`. Then:

- Define `ex12_star` as `ex12_p * ex12_q` -- the entry-by-entry product, not the matrix
  product. Print it next to `ex12_p @ ex12_q` and look at how far apart they are.
- Define `ex12_assoc` as what `np.allclose` reports when you compare the two bracketings
  $(PQ)R$ and $P(QR)$.

$P$ and $Q$ are the pair from Exercise 5, so `ex12_p @ ex12_q` is also a free machine mark
on that hand answer. Print it and compare -- if it disagrees with what you wrote there, the
hand version is the one to revisit.

<details><summary>Hint 1 — nudge</summary>

Section 5 is the whole exercise: `@` and `*` are different operations that happily accept
the same arguments, and Section 4 promised that regrouping a chain of products changes
nothing. Both claims are one line of NumPy away from being settled.

</details>
<details><summary>Hint 2 — approach</summary>

Build the three matrices with `np.array` and nested lists. For associativity, compute the
two bracketings separately -- brackets around the first pair, then brackets around the
second -- and pass both to `np.allclose`, which is how you compare arrays of floats without
tripping over representation.

</details>


In [ ]:
# Exercise 12: Verify a hand result, contrast * with @, and test associativity in NumPy.
# Define: ex12_p, ex12_q, ex12_r, ex12_star, ex12_assoc.

# TODO: your code here


In [ ]:
# Check Exercise 12 -- run after your attempt.
from lib.grading import check

with check("Exercise 12"):
    assert (
        np.asarray(ex12_p).shape == (2, 2)
        and np.asarray(ex12_q).shape == (2, 2)
        and np.asarray(ex12_r).shape == (2, 2)
    ), "all three matrices in this exercise are 2x2"
    p12 = np.asarray(ex12_p)
    q12 = np.asarray(ex12_q)
    r12 = np.asarray(ex12_r)
    assert (
        np.allclose(p12, [[2, 0], [1, 3]])
        and np.allclose(q12, [[1, 4], [2, -1]])
        and np.allclose(r12, [[1, 1], [0, 2]])
    ), "build P, Q and R exactly as the prompt lists them, row by row"
    assert np.asarray(ex12_star).shape == (2, 2), (
        "the entry-by-entry product of two 2x2 matrices is 2x2 as well"
    )
    assert np.allclose(np.asarray(ex12_star), p12 * q12), (
        "ex12_star is the entry-by-entry product -- position by position, with no sums"
    )
    assert not np.allclose(np.asarray(ex12_star), p12 @ q12), (
        "if * and @ agreed here, one of the two was computed with the other operator"
    )
    assert bool(ex12_assoc) == bool(np.allclose((p12 @ q12) @ r12, p12 @ (q12 @ r12))), (
        "ex12_assoc should record what np.allclose actually reports for your two bracketings"
    )
    assert bool(ex12_assoc), (
        "regrouping a chain of products cannot change it -- if you got False, check that "
        "both bracketings use @ and keep the three matrices in the same order"
    )


### Solutions


In [ ]:
# --- Exercise 1 ---
ex1_iv = [4, -3]                       # 1*4 + 0*(-3), then 0*4 + 1*(-3)
print("I x =", ex1_iv)

# --- Exercise 2 ---
ex2_sv = [-3, 4]                       # row [0, 1] takes the second entry; row [1, 0] the first
print("S x =", ex2_sv)

# --- Exercise 3 ---
ex3_mv = [8, 23]                       # 2*5 + (-1)*2 = 8;  3*5 + 4*2 = 23
print("N u =", ex3_mv)

# --- Exercise 4 ---
ex4_product = [[2, 1], [4, 3]]         # K swaps J's columns
print("J K =", ex4_product)

# --- Exercise 5 ---
ex5_product = [[2, 8], [7, 1]]         # 2*1 + 0*2 = 2;  2*4 + 0*(-1) = 8;  1*1 + 3*2 = 7;  1*4 + 3*(-1) = 1
print("P Q =", ex5_product)

# --- Exercise 6 ---
ex6_product = [[5, 15], [0, -10]]      # (-2)*2 + 4*1 = 0 is the one that catches people
print("E F =", ex6_product)

# --- Exercise 7 ---
ex7_tw = [8, -6, 7]                    # 1*2 + 0*(-1) + 2*3 = 8, and so on down the rows
print("T w =", ex7_tw)

# --- Exercise 8 ---
ex8_product = [[3, 1, 1], [0, 2, 6], [2, 2, 3]]
print("G H =", ex8_product)

# --- Exercise 9 ---
ex9_uv = [[5, 2], [2, 13]]             # (2, 3) times (3, 2) -> (2, 2), three-term sums
ex9_vu = [[1, 2, 0], [2, 5, 3], [0, 4, 12]]   # (3, 2) times (2, 3) -> (3, 3), two-term sums
print("U V =", ex9_uv)
print("V U =", ex9_vu)

# --- Exercise 10 ---
ex10_shapes = {
    "a": (2, 4),        # (2, 3)(3, 4): inner 3 and 3 agree
    "b": None,          # (3, 4)(2, 3): inner 4 and 2 disagree
    "c": (4, 4),        # (4, 1)(1, 4): one shared column, and the answer grows
    "d": (1, 1),        # (1, 4)(4, 1): the same two factors the other way round
    "e": None,          # (2, 2)(3, 3): inner 2 and 3 disagree
}
print("ex10_shapes =", ex10_shapes)

# --- Exercise 11 ---
ex11_cd = [[2, 1], [1, 1]]
ex11_dc = [[1, 1], [1, 2]]
ex11_commute = False                   # same two matrices, two different products
print("C D =", ex11_cd, "  D C =", ex11_dc, "  commute?", ex11_commute)

# --- Exercise 12 ---
ex12_p = np.array([[2, 0], [1, 3]])
ex12_q = np.array([[1, 4], [2, -1]])
ex12_r = np.array([[1, 1], [0, 2]])
ex12_star = ex12_p * ex12_q            # entry by entry -- nothing like the matrix product
ex12_assoc = np.allclose((ex12_p @ ex12_q) @ ex12_r, ex12_p @ (ex12_q @ ex12_r))
print("P @ Q =", (ex12_p @ ex12_q).tolist(), "   P * Q =", ex12_star.tolist())
print("associative?", ex12_assoc)


## 8. Where this lands: a gate is a matrix

Everything above is the machinery of quantum computing, not a warm-up for it.

A single qubit's state is a two-entry column vector. The two states you can be certain
about are written

$$\lvert 0 \rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \qquad
\lvert 1 \rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$$

A **gate** is a matrix, and applying a gate to a qubit is the matrix-times-column-vector
product from Section 1. Nothing more exotic than that happens.

Now look again at the swap matrix from the very first worked example. Under its own name it
is the **Pauli X gate**, the quantum NOT:

$$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \qquad
X \lvert 0 \rangle = \lvert 1 \rangle, \qquad
X \lvert 1 \rangle = \lvert 0 \rangle$$

Both of those are Exercise 2 with different numbers in the vector. Multiplying $X$ by the
column $(1, 0)$ leaves the top entry at zero, while the second row's single 1 picks the
vector's 1 up into the bottom slot. Out comes $(0, 1)$: a qubit that was definitely 0 is
now definitely 1.

Apply two gates in a row and you get a matrix product. Running $U$ first and then $V$ sends
a state $\lvert \psi \rangle$ to $V(U \lvert \psi \rangle)$, and associativity --
Section 4, and Exercise 12 -- says you may as well multiply the two gates together first
and apply the single matrix $VU$. That is why an entire circuit collapses into one matrix,
and why a simulator builds that matrix instead of replaying the gates one at a time.

Note the order carefully: the gate that runs **first** sits on the **right**, next to the
state it acts on. And now read Section 4 once more, because it has become a statement about
physics rather than about arithmetic. $UV \ne VU$ means **the order you apply gates in
changes what comes out**. Rotating a qubit one way and then another is a different
experiment from doing it in the opposite order, and the eight dot products you ground
through in Exercise 11 are the reason why.


## Summary

- **A matrix times a column vector is one dot product per row.** Row in, number out, so
  the matrix's row count fixes the answer's length.
- **A matrix times a matrix is that same job done once per column of the right-hand
  factor.** Entry by entry, $(AB)_{ij} = \sum_k A_{ik} B_{kj}$: row $i$ of the left
  against column $j$ of the right.
- **Shapes decide whether there is a product at all.**
  $(m \times n)(n \times p) = (m \times p)$ -- the inner dimensions must match and then
  vanish into the sum, the outer ones survive. Mismatched inner dimensions mean no product,
  not a zero and not an error to work around.
- **Order matters, grouping does not.** $AB \ne BA$ in general, while $(AB)C = A(BC)$
  always. The identity is the multiplicative one and the zero matrix is the multiplicative
  zero -- though unlike ordinary numbers, two nonzero matrices can still multiply to zero.
- **In NumPy `@` is the matrix product and `*` is entry by entry.** Both accept the same
  arguments and neither one warns you. Print `.shape` when a result surprises you, and
  remember that a 1-D array lets `@` choose its orientation on your behalf.
- **A gate is a matrix and a circuit is a product of gates**, which is why the
  non-commutativity you proved by hand in Exercise 11 is a fact about hardware and not a
  quirk of notation.

**You finished notebook 3.** Carry the shape rule with you: everything else in this section
is built on the product you just learned to do without a computer.
